# D2_03 ANOVA · PCA · 요인분석

> **오늘 질문:** 제품 등급에 따라 공정이 다른가? 590개 센서를 몇 개의 축으로 요약할 수 있을까?

| 파트 | 데이터 | 방법 |
|---|---|---|
| 1 | 공장 분석 테이블 | 등급(L·M·H)별 ANOVA + 사후검정 |
| 2 | SECOM (반도체 센서 590개) | PCA |
| 3 | Steel Plates (강판 결함) | 요인분석 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from factor_analyzer import FactorAnalyzer

df = pd.read_parquet("../../data/factory_table.parquet")
secom = pd.read_csv("../../data/secom.csv")
steel = pd.read_csv("../../data/steel_plates.csv")

## 1. 등급별 ANOVA: 토크 ✍️

세 그룹을 t-test로 여러 번 비교하면 '우연한 유의'가 늘어납니다. ANOVA로 한 번에 검정합니다.

> 변수 이름: `L`, `M`, `H`

In [ ]:
# 힌트: stats.f_oneway(그룹1, 그룹2, 그룹3)

## 2. 등급별 ANOVA: 진동 ✍️

같은 방법으로 진동을 비교하고, 등급별 평균도 함께 봅니다.

## 3. 사후검정: 어디가 다른가 ✍️

ANOVA가 유의하면 Tukey HSD로 어느 쌍이 다른지 봅니다. `reject`가 True인 쌍이 차이가 있는 쌍입니다.

In [ ]:
# 힌트: pairwise_tukeyhsd(값, 그룹)

**Q1.** 토크와 진동의 ANOVA 결과를 각각 해석해 보세요. 진동 차이는 현장에서 중요할까요?

> 답:


## 4. SECOM 전처리 📋

PCA 전에 ① 결측을 채우고 ② 표준화(평균 0, 표준편차 1)합니다. 단위가 다른 센서를 공평하게 다루기 위해서입니다.

In [ ]:
X = secom.drop(columns=["timestamp", "defect"])
print(f"센서 {X.shape[1]}개, 결측 {X.isna().sum().sum():,}칸")

X = X.fillna(X.median())                       # 결측은 각 센서의 중앙값으로
X_scaled = StandardScaler().fit_transform(X)   # 표준화

## 5. PCA와 누적 설명 분산 ✍️

`explained_variance_ratio_`는 각 주성분이 담은 정보의 비율입니다. `cumsum()`으로 누적합니다.

> 변수 이름: `pca`, `cum`

In [ ]:
# 힌트: PCA().fit(데이터), pca.explained_variance_ratio_.cumsum()

누적 80%에 도달하는 주성분 개수를 셉니다.

**Q2.** 590개 센서를 몇 개의 주성분으로 요약할 수 있나요? PC1 하나가 담은 정보는 얼마나 되나요?

> 답:


## 6. 요인분석: 강판 결함 데이터 📋

강판 결함의 크기·둘레·밝기 관련 변수 7개로 요인 2개를 찾습니다. 결과 표(요인 적재량)를 읽는 것이 목표입니다.

In [ ]:
cols = ["pixels_areas", "x_perimeter", "y_perimeter", "sum_of_luminosity",
        "minimum_of_luminosity", "maximum_of_luminosity", "luminosity_index"]

fa = FactorAnalyzer(n_factors=2, rotation="varimax", method="principal").fit(steel[cols])
eigen, _ = fa.get_eigenvalues()
print("고유값:", eigen.round(2))

loadings = pd.DataFrame(fa.loadings_, index=cols, columns=["요인1", "요인2"])
loadings.round(2)

**Q3.** 적재량이 0.4 이상인 변수를 보고 요인1과 요인2에 이름을 붙여 보세요. 고유값으로 보면 요인 2개가 적절한가요?

> 답:
